# German Language Pipeline Adaptation
This notebook adapts the word processing pipeline from [this code](https://github.com/t-redactyl/language-learning-agent) for German.

Word list captured from [Github](https://github.com/eymenefealtun/all-words-in-all-languages).

In [ ]:
import os
from string import punctuation
import pandas as pd
import spacy
from wordfreq import zipf_frequency

LANG_CONFIG = {
    "german": {"model": "de_core_news_lg", "freq_code": "de"},
#    "english": {"model": "en_core_web_lg", "freq_code": "en"},
#    "spanish": {"model": "es_core_news_lg", "freq_code": "es"}
}

/home/heitor/miniforge3/envs/language_agent/lib/python3.13/site-packages/torch/cuda/__init__.py:180: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12060). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [ ]:
def load_and_clean_word_list(language: str) -> pd.DataFrame:
    """Loads a comma-separated word list from data/{language}/{language}.txt"""
    
    path = f"data/{language}/{language}.txt"
    if not os.path.exists(path):
        raise FileNotFoundError(f"Expected file at {path}")
        
    with open(path, "r", encoding="utf-8") as f:
        word_list = f.read().split(",")

    word_df = pd.DataFrame({"word": word_list})
    # Remove whitespace and punctuation
    word_df["word"] = word_df["word"].str.strip().str.strip(punctuation)
    # Remove empty strings
    word_df = word_df[word_df["word"] != ""]
    
    return word_df

def add_lemma(df: pd.DataFrame, nlp, batch_size: int = 1000) -> pd.DataFrame:
    """Uses spaCy to find the root form (lemma) of words."""

    docs = nlp.pipe(df["word"].astype(str).tolist(), batch_size=batch_size)
    lemmas = [doc[0].lemma_ if len(doc) > 0 else "" for doc in docs]
    df["lemma"] = lemmas
    return df

def add_word_frequencies(df: pd.DataFrame, language: str) -> pd.DataFrame:
    """Adds Zipf frequency scores using the correct ISO language code."""
    
    lang_key = language.lower()
    freq_code = LANG_CONFIG.get(lang_key, {}).get("freq_code", "en")
    
    df["zipf_freq_lemma"] = [zipf_frequency(str(w), freq_code) for w in df["lemma"]]
    return df

def clean_up_and_export(df: pd.DataFrame, language: str) -> None:
    """Filters duplicates, assigns difficulty levels, and saves to JSON."""
    
    # Keep the version of the word with the highest frequency score
    df = (
        df.loc[df.groupby("lemma", sort=False)["zipf_freq_lemma"].idxmax()]
        .reset_index(drop=True)
    )

    # Remove words with zero frequency (likely noise/typos)
    df = df[df["zipf_freq_lemma"] > 0].copy()

    # Assign difficulty based on Zipf scale (0-8)
    df.loc[:, "word_difficulty"] = pd.cut(
        df["zipf_freq_lemma"],
        bins = [-float("inf"), 3.0, 5.0, float("inf")],
        labels = ["advanced", "intermediate", "beginner"],
        include_lowest = True
    )

    output_df = df.drop(columns=["word", "zipf_freq_lemma"])
    output_df = output_df.rename(columns={"lemma": "word"})

    os.makedirs(f"data/{language}", exist_ok=True)
    output_df.to_json(f"data/{language}/word-list-cleaned.json", orient="index", force_ascii=False)
    print(f"Exported cleaned {language} list.")

In [3]:
def create_clean_word_list(language: str) -> None:
    lang_key = language.lower()
    if lang_key not in LANG_CONFIG:
        print(f"Language {language} not supported.")
        return

    model_name = LANG_CONFIG[lang_key]["model"]
    try:
        nlp = spacy.load(model_name, disable=["parser", "ner", "textcat"])
    except OSError:
        print(f"Model {model_name} not found. Please run: python -m spacy download {model_name}")
        return

    print(f"Processing: {language}")
    lang_df = load_and_clean_word_list(lang_key)
    lang_df = add_lemma(lang_df, nlp)
    lang_df = add_word_frequencies(lang_df, lang_key)
    clean_up_and_export(lang_df, lang_key)

create_clean_word_list("German")

/home/heitor/miniforge3/envs/language_agent/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing: German
Exported cleaned german list.
